# Shot Suppression / Block Modelling

This notebook implements three shot-suppression metrics for WSL/Eredivisie data:

| Option | Name | Description |
|--------|------|-------------|
| 1 | **xG Blocked** | Raw xG credit for each blocked shot |
| 2 | **Blocks Above Expected (BAE)** | Actual blocks minus model-predicted blocks |
| 3 | **xGS+ (xG Suppressed Plus)** | xG-weighted suppression above expectation |

Plus a composite **Shot Suppression Score** combining blocked and non-blocked xGA.

**Block type split (via Opta Q82):**
- `is_outfield_block = 1` — typeId=15 with qualifier 82 → outfield defender block
- `is_keeper_save = 1`    — typeId=15 without qualifier 82 → goalkeeper save
- `is_blocked = 1`        — combined (legacy, = outfield block OR keeper save)

Section 4b runs a dedicated **pure outfield block model** using `is_outfield_block` as the target, isolating defensive-line block skill from goalkeeper performance.

**Data**: `all_shots_full.csv`. `is_outfield_block` / `is_keeper_save` columns require data loaded with the updated `load_match` function.

---
## Section 1 — Setup

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'xgboost', 'scikit-learn',
                'matplotlib', 'seaborn', 'shap'], check=True)
print('Dependencies ready.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve

import shap

# ── CONFIG ────────────────────────────────────────────────────────────────────
CSV_PATH  = 'all_shots_full.csv'   # place file alongside this notebook to run locally
LAMBDA    = 0.35                   # weight for blocked xGA in composite score
SEED      = 42

# ── COLOURS ───────────────────────────────────────────────────────────────────
NAVY   = '#0D1B2A'
TEAL   = '#1B7A78'
GOLD   = '#E8C33A'
RED    = '#E05252'
GREEN  = '#3fb950'
GREY   = '#8B9DB0'

# ── MATPLOTLIB STYLE ──────────────────────────────────────────────────────────
plt.style.use('dark_background')
matplotlib.rcParams.update({
    'figure.facecolor' : NAVY,
    'axes.facecolor'   : '#111d2b',
    'axes.edgecolor'   : GREY,
    'axes.labelcolor'  : 'white',
    'xtick.color'      : GREY,
    'ytick.color'      : GREY,
    'text.color'       : 'white',
    'grid.color'       : '#1e2d3d',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.5,
    'font.family'      : 'DejaVu Sans',
})

print('Setup complete.')

---
## Section 2 — Load & Explore

In [ ]:
df = pd.read_csv(CSV_PATH, low_memory=False)
print(f'Loaded {len(df):,} shots, {df.shape[1]} columns')
print(f'Teams: {df["contestant_id"].nunique()}')
df.head(3)

In [ ]:
# ── Blocked vs non-blocked breakdown ──────────────────────────────────────────
breakdown = df.groupby('is_blocked').agg(
    count=('xg', 'count'),
    mean_xg=('xg', 'mean'),
    total_xg=('xg', 'sum'),
    goals=('is_goal', 'sum'),
).rename(index={0: 'Not Blocked', 1: 'Blocked'})
breakdown['conversion_rate'] = (breakdown['goals'] / breakdown['count'] * 100).round(2)
breakdown['pct_of_shots']    = (breakdown['count'] / len(df) * 100).round(1)
print('=== Combined block breakdown ===')
display(breakdown)

# ── Outfield block vs keeper save split (requires Q82-updated data) ───────────
if 'is_outfield_block' in df.columns and 'is_keeper_save' in df.columns:
    ob_count  = df['is_outfield_block'].sum()
    ks_count  = df['is_keeper_save'].sum()
    tot_block = df['is_blocked'].sum()
    print(f'\n=== Block-type split (Opta Q82) ===')
    print(f'Outfield blocks  : {ob_count:,}  ({ob_count/tot_block*100:.1f}% of all typeId=15)')
    print(f'Keeper saves     : {ks_count:,}  ({ks_count/tot_block*100:.1f}% of all typeId=15)')
    ob_split = df.groupby('is_outfield_block').agg(
        count=('xg','count'), mean_xg=('xg','mean'), goals=('is_goal','sum')
    )
    ob_split['conv'] = (ob_split['goals'] / ob_split['count'] * 100).round(2)
    display(ob_split.rename(index={0: 'Keeper Save', 1: 'Outfield Block'}))
else:
    print('\nis_outfield_block column not found — re-load CSV from updated load_match to enable split.')

In [ ]:
# ── xG distribution plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('xG Distribution: Blocked vs Non-Blocked Shots', fontsize=14, color='white', y=1.02)

# Histogram
ax = axes[0]
bins = np.linspace(0, 1, 40)
ax.hist(df.loc[df['is_blocked']==0, 'xg'], bins=bins, color=RED,   alpha=0.7, label='Not Blocked', density=True)
ax.hist(df.loc[df['is_blocked']==1, 'xg'], bins=bins, color=TEAL,  alpha=0.7, label='Blocked',     density=True)
ax.set_xlabel('xG')
ax.set_ylabel('Density')
ax.set_title('xG Distribution by Block Status')
ax.legend()
ax.grid(True)

# Summary bar
ax2 = axes[1]
labels  = ['Not Blocked\n(n=1,929)', 'Blocked\n(n=2,138)']
mean_xg = [df.loc[df['is_blocked']==0, 'xg'].mean(), df.loc[df['is_blocked']==1, 'xg'].mean()]
bars = ax2.bar(labels, mean_xg, color=[RED, TEAL], width=0.5, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, mean_xg):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=11)
ax2.set_ylabel('Mean xG')
ax2.set_title('Mean xG by Block Status')
ax2.grid(True, axis='y')

plt.tight_layout()
plt.show()

---
## Section 3 — Option 1: xG Blocked (Simple)

The simplest shot-suppression credit: **for every blocked shot, credit the defending team with the xG of that shot.**

This captures the raw xG value removed from the opposition's attacking threat. A team with high `xg_blocked` is regularly preventing dangerous shot attempts from reaching the keeper.

**Limitations**: treats all blocks equally regardless of whether a block was expected given shot location/type. A team facing many low-quality shots will accumulate xG-blocked even if they are poor at blocking.

**Formula**: `xG_Blocked_Team = Σ xG_i × I(blocked_i)` for all shots faced by that team.

In [ ]:
# ── Team-level xG blocked ──────────────────────────────────────────────────────
team_stats = df.groupby('contestant_id').agg(
    shots_faced    = ('xg', 'count'),
    blocked        = ('is_blocked', 'sum'),
    goals_conceded = ('is_goal', 'sum'),
    xg_conceded    = ('xg', 'sum'),
    xg_blocked     = ('xg', lambda x: x[df.loc[x.index, 'is_blocked'] == 1].sum()),
    block_rate     = ('is_blocked', 'mean'),
).reset_index()

team_stats['non_blocked_xga'] = team_stats['xg_conceded'] - team_stats['xg_blocked']
team_stats = team_stats.sort_values('xg_blocked', ascending=False)

display(team_stats[['contestant_id','shots_faced','blocked','block_rate',
                     'xg_conceded','xg_blocked','non_blocked_xga','goals_conceded']].round(3))

In [ ]:
# ── Horizontal bar chart: xG Blocked by team ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

sorted_stats = team_stats.sort_values('xg_blocked')
bars = ax.barh(sorted_stats['contestant_id'].astype(str),
               sorted_stats['xg_blocked'],
               color=TEAL, edgecolor='white', linewidth=0.4, alpha=0.9)

for bar, val in zip(bars, sorted_stats['xg_blocked']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', ha='left', fontsize=8, color=GREY)

ax.set_xlabel('xG Blocked (sum of blocked-shot xG)')
ax.set_title('Option 1 — xG Blocked by Team\n(Higher = more xG removed from opposition)')
ax.grid(True, axis='x')
plt.tight_layout()
plt.show()

---
## Section 4 — Option 2: Blocks Above Expected (BAE)

**BAE removes the confound of shot quality from the raw block count.**

We first train an XGBoost model to estimate the probability that any given shot will be blocked, based on shot characteristics (location, type, context). Then:

$$\text{BAE} = \sum_{i=1}^{N} \left( \text{Blocked}_i - P(\text{block}_i) \right)$$

- **Positive BAE**: team blocked more shots than the model predicted — genuine defensive over-performance.
- **Negative BAE**: team blocked fewer shots than expected — under-performance relative to the average team facing the same shots.

We use `GroupShuffleSplit` on `match_file` to avoid data leakage (all shots from a single match stay in train or test).

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
df = df.copy()

# Symmetrise y around pitch centre (0–68 → 0–34 mirrored)
if 'y_sym' not in df.columns:
    df['y_sym'] = (df['y'] - 34).abs() if 'y' in df.columns else 0

# Log distance
if 'distance' in df.columns:
    df['log_distance'] = np.log1p(df['distance'])
else:
    df['distance']     = np.sqrt((df['x'] - 100)**2 + (df['y'] - 34)**2)
    df['log_distance'] = np.log1p(df['distance'])

# Angle sin
if 'angle' in df.columns:
    df['angle_sin'] = np.sin(np.radians(df['angle']))
else:
    df['angle']     = 0
    df['angle_sin'] = 0

# Zone flags
df['in_six_yard']    = ((df['x'] >= 94) & (df['y_sym'] <= 5)).astype(int)
df['in_penalty_box'] = ((df['x'] >= 83) & (df['y_sym'] <= 20)).astype(int)
df['central_y']      = (df['y_sym'] <= 10).astype(int)

# Set-piece / pull-back
for col in ['is_set_piece', 'is_pull_back']:
    if col not in df.columns:
        df[col] = 0

BLOCK_FEATURES = [
    'x', 'y_sym', 'distance', 'log_distance', 'angle', 'angle_sin',
    'in_six_yard', 'in_penalty_box', 'central_y',
    'is_header', 'is_right_foot', 'is_left_foot', 'weak_foot',
    'is_volley', 'is_deflected', 'is_first_time', 'is_big_chance',
    'is_fast_break', 'is_from_corner', 'is_free_kick', 'is_set_piece',
    'is_open_play', 'is_pull_back', 'under_pressure', 'xg'
]

# Keep only features that actually exist in df
BLOCK_FEATURES = [f for f in BLOCK_FEATURES if f in df.columns]
print(f'Using {len(BLOCK_FEATURES)} features:', BLOCK_FEATURES)

In [ ]:
# ── Train / test split (by match to avoid leakage) ────────────────────────────
X_all = df[BLOCK_FEATURES].fillna(0).values
y_all = df['is_blocked'].values

# Determine groups
group_col = 'match_file' if 'match_file' in df.columns else 'contestant_id'
groups    = df[group_col].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X_all, y_all, groups))

X_train, X_test = X_all[train_idx], X_all[test_idx]
y_train, y_test = y_all[train_idx], y_all[test_idx]

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# ── Train XGBoost block model ─────────────────────────────────────────────────
block_model = XGBClassifier(
    n_estimators       = 300,
    max_depth          = 4,
    learning_rate      = 0.05,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    use_label_encoder  = False,
    eval_metric        = 'logloss',
    random_state       = SEED,
    verbosity          = 0,
)
block_model.fit(X_train, y_train)

y_pred_proba = block_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f'Test ROC-AUC: {auc:.4f}')

In [ ]:
# ── Calibration plot ──────────────────────────────────────────────────────────
fraction_pos, mean_predicted = calibration_curve(y_test, y_pred_proba, n_bins=10)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1],     color=GREY,  linestyle='--', label='Perfect calibration')
ax.plot(mean_predicted, fraction_pos, 'o-', color=TEAL, linewidth=2, label=f'XGBoost (AUC={auc:.3f})')
ax.set_xlabel('Mean Predicted P(block)')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Block Model Calibration Plot')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── Predict P(block) for all shots ────────────────────────────────────────────
df['p_block']     = block_model.predict_proba(X_all)[:, 1]
df['bae_contrib'] = df['is_blocked'] - df['p_block']  # per-shot BAE contribution

team_bae = df.groupby('contestant_id').agg(
    bae              = ('bae_contrib', 'sum'),
    p_block_mean     = ('p_block', 'mean'),
    actual_block_rate = ('is_blocked', 'mean'),
).reset_index().sort_values('bae', ascending=False)

display(team_bae.round(3))

In [ ]:
# ── BAE bar chart ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

sorted_bae = team_bae.sort_values('bae')
colours    = [GREEN if v >= 0 else RED for v in sorted_bae['bae']]

bars = ax.barh(sorted_bae['contestant_id'].astype(str),
               sorted_bae['bae'],
               color=colours, edgecolor='white', linewidth=0.4, alpha=0.9)

ax.axvline(0, color='white', linewidth=1.2, linestyle='-')

for bar, val in zip(bars, sorted_bae['bae']):
    offset = 0.5 if val >= 0 else -0.5
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}', va='center', ha='left' if val >= 0 else 'right',
            fontsize=8, color=GREY)

ax.set_xlabel('Blocks Above Expected (BAE)')
ax.set_title('Option 2 — Blocks Above Expected (BAE) by Team\nPositive = blocked more than model predicted')
ax.grid(True, axis='x')

pos_patch = mpatches.Patch(color=GREEN, label='Over-performed (BAE > 0)')
neg_patch = mpatches.Patch(color=RED,   label='Under-performed (BAE < 0)')
ax.legend(handles=[pos_patch, neg_patch], loc='lower right')

plt.tight_layout()
plt.show()

---
## Section 4b — Pure Outfield Block Model (Q82 split)

The combined `is_blocked` target confounds **outfield-defender blocks** with **goalkeeper saves**. Using the Q82 qualifier we can isolate pure outfield blocks.

This section trains a separate XGBoost model with `is_outfield_block` as target (excluding keeper saves entirely), producing:
- `p_outfield_block` — probability that an outfield player blocks this shot
- `ob_bae` — Outfield-Block BAE: outfield blocks above expected, per team

A team with high `ob_bae` has defenders who are genuinely over-performing at blocking relative to shot characteristics — this is a cleaner signal of defensive-line quality than the combined BAE.

In [ ]:
if 'is_outfield_block' not in df.columns:
    print('WARNING: is_outfield_block not in data. Re-export CSV with updated load_match.')
    print('Skipping Section 4b.')
else:
    # ── Train pure outfield block model ──────────────────────────────────────
    y_ob     = df['is_outfield_block'].values
    group_col_ob = 'match_file' if 'match_file' in df.columns else 'contestant_id'
    groups_ob    = df[group_col_ob].values

    gss_ob = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    tr_ob, te_ob = next(gss_ob.split(X_all, y_ob, groups_ob))

    ob_model = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=SEED, verbosity=0,
    )
    ob_model.fit(X_all[tr_ob], y_ob[tr_ob])

    ob_pred_proba = ob_model.predict_proba(X_all[te_ob])[:, 1]
    ob_auc = roc_auc_score(y_ob[te_ob], ob_pred_proba)
    print(f'Outfield Block Model — Test ROC-AUC: {ob_auc:.4f}')
    print(f'  Base rate (outfield blocks): {y_ob.mean()*100:.1f}% of all shots')

    # ── Predict P(outfield block) for all shots ───────────────────────────────
    df['p_outfield_block'] = ob_model.predict_proba(X_all)[:, 1]
    df['ob_bae_contrib']   = df['is_outfield_block'] - df['p_outfield_block']

    team_ob_bae = df.groupby('contestant_id').agg(
        ob_bae              = ('ob_bae_contrib', 'sum'),
        p_ob_mean           = ('p_outfield_block', 'mean'),
        actual_ob_rate      = ('is_outfield_block', 'mean'),
        keeper_save_rate    = ('is_keeper_save', 'mean'),
    ).reset_index().sort_values('ob_bae', ascending=False)

    print('\nOutfield Block BAE by team:')
    display(team_ob_bae.round(3))

    # ── OB-BAE bar chart ──────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # OB BAE
    ax1 = axes[0]
    sorted_ob = team_ob_bae.sort_values('ob_bae')
    col_ob    = [GREEN if v >= 0 else RED for v in sorted_ob['ob_bae']]
    bars1 = ax1.barh(sorted_ob['contestant_id'].astype(str), sorted_ob['ob_bae'],
                     color=col_ob, edgecolor='white', linewidth=0.4, alpha=0.9)
    ax1.axvline(0, color='white', linewidth=1.2)
    for bar, val in zip(bars1, sorted_ob['ob_bae']):
        ax1.text(val + (0.3 if val >= 0 else -0.3), bar.get_y() + bar.get_height()/2,
                 f'{val:+.1f}', va='center', ha='left' if val >= 0 else 'right',
                 fontsize=8, color=GREY)
    ax1.set_xlabel('Outfield-Block BAE')
    ax1.set_title('Outfield Block BAE by Team\n(Q82 split — excludes keeper saves)')
    ax1.grid(True, axis='x')

    # Combined BAE vs OB BAE scatter
    ax2 = axes[1]
    merged_bae = team_bae.merge(team_ob_bae[['contestant_id','ob_bae']], on='contestant_id')
    ax2.scatter(merged_bae['bae'], merged_bae['ob_bae'],
                color=GOLD, s=80, edgecolors='white', linewidth=0.6, zorder=3)
    for _, row in merged_bae.iterrows():
        ax2.annotate(str(row['contestant_id'])[:8],
                     (row['bae'], row['ob_bae']),
                     textcoords='offset points', xytext=(4, 4), fontsize=7, color=GREY)
    ax2.axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
    ax2.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
    ax2.set_xlabel('Combined BAE (outfield + keeper)')
    ax2.set_ylabel('Outfield-Only BAE (Q82)')
    ax2.set_title('Combined vs Outfield-Only BAE\n(Divergence = keeper vs outfield contribution)')
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

    print('\nInterpretation: teams in Q2 (top-left) have good outfield blockers but poor combined BAE')
    print('— suggests the keeper is dragging down the combined metric.')

---
## Section 5 — Option 3: xGS+ (Expected xG Suppressed — Best)

**xGS+ (xG Suppressed Plus)** is the most complete metric. It combines both the *quality* of the blocked shots (via xG) and the *difficulty* of blocking them (via P(block)).

**Per-shot components**:
- `expected_suppression_i = xG_i × P(block_i)` — what an average team would be expected to suppress on this shot
- `actual_suppression_i   = xG_i × I(blocked_i)` — what was actually suppressed

**Team-level formula**:

$$\text{xGS+} = \sum_i xG_i \cdot I(\text{blocked}_i) - \sum_i xG_i \cdot P(\text{block}_i)$$

- **Positive xGS+**: team suppressed more xG than expected — elite defensive blocking.
- **Negative xGS+**: team suppressed less xG than expected — poor blocking on high-quality chances.

This is the **recommended metric** because it rewards teams that block high-xG shots that are hard to block.

In [ ]:
# ── xGS+ calculation ─────────────────────────────────────────────────────────
df['expected_suppression'] = df['xg'] * df['p_block']
df['actual_suppression']   = df['xg'] * df['is_blocked']
df['xgs_contrib']          = df['actual_suppression'] - df['expected_suppression']

team_xgs = df.groupby('contestant_id').agg(
    xgs_plus             = ('xgs_contrib',          'sum'),
    expected_suppression = ('expected_suppression', 'sum'),
    actual_suppression   = ('actual_suppression',   'sum'),
    shots_faced          = ('xg',                   'count'),
).reset_index()

team_xgs['xgs_per_shot'] = team_xgs['xgs_plus'] / team_xgs['shots_faced']
team_xgs = team_xgs.sort_values('xgs_plus', ascending=False)

display(team_xgs.round(3))

In [ ]:
# ── xGS+ bar chart ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart with 0-line
ax1 = axes[0]
sorted_xgs = team_xgs.sort_values('xgs_plus')
colours    = [GREEN if v >= 0 else RED for v in sorted_xgs['xgs_plus']]

bars = ax1.barh(sorted_xgs['contestant_id'].astype(str),
                sorted_xgs['xgs_plus'],
                color=colours, edgecolor='white', linewidth=0.4, alpha=0.9)
ax1.axvline(0, color='white', linewidth=1.2)

for bar, val in zip(bars, sorted_xgs['xgs_plus']):
    offset = 0.05 if val >= 0 else -0.05
    ax1.text(val + offset, bar.get_y() + bar.get_height()/2,
             f'{val:+.2f}', va='center', ha='left' if val >= 0 else 'right',
             fontsize=8, color=GREY)

ax1.set_xlabel('xGS+ (xG Suppressed Plus)')
ax1.set_title('Option 3 — xGS+ by Team\nPositive = suppressed more xG than expected')
ax1.grid(True, axis='x')

# Scatter: expected vs actual suppression
ax2 = axes[1]
ax2.scatter(sorted_xgs['expected_suppression'], sorted_xgs['actual_suppression'],
            color=GOLD, s=80, edgecolors='white', linewidth=0.5, zorder=3)

# Diagonal reference line
lim = max(sorted_xgs['expected_suppression'].max(), sorted_xgs['actual_suppression'].max()) * 1.05
ax2.plot([0, lim], [0, lim], color=GREY, linestyle='--', linewidth=1.2, label='Expected = Actual')

for _, row in sorted_xgs.iterrows():
    ax2.annotate(str(row['contestant_id'])[:8],
                 (row['expected_suppression'], row['actual_suppression']),
                 textcoords='offset points', xytext=(4, 4), fontsize=7, color=GREY)

ax2.set_xlabel('Expected Suppression (Σ xG × P(block))')
ax2.set_ylabel('Actual Suppression (Σ xG × I(blocked))')
ax2.set_title('Expected vs Actual xG Suppression\n(Above diagonal = positive xGS+)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

---
## Section 6 — Team-Level Shot Suppression Score

A composite metric that combines **non-blocked xGA** with a discounted **xG-blocked** term:

$$\text{Suppression Score} = \text{Non-Blocked xGA} + \lambda \times \text{xG\_Blocked}$$

where **λ = 0.35**. This reflects the fact that a blocked shot still represents an allowed opportunity — the opposition got to the shot — but carries less danger than an unblocked attempt.

- **Lower score = better defensive suppression.**
- Unlike raw xGA, this penalises teams that allow high-quality shots even when they happen to get blocked.
- λ = 0.35 is a calibrated weight; teams can experiment with different values.

In [ ]:
# ── Composite Suppression Score ───────────────────────────────────────────────
team_merged = team_stats.merge(
    team_xgs[['contestant_id', 'xgs_plus']], on='contestant_id'
)
team_merged = team_merged.merge(
    team_bae[['contestant_id', 'bae']], on='contestant_id'
)

team_merged['suppression_score'] = (
    team_merged['non_blocked_xga'] + LAMBDA * team_merged['xg_blocked']
)
team_merged['rank'] = team_merged['suppression_score'].rank().astype(int)
team_merged = team_merged.sort_values('suppression_score')

display(team_merged[['contestant_id','rank','xg_conceded','suppression_score',
                      'xg_blocked','non_blocked_xga','bae','xgs_plus']].round(3))

In [ ]:
# ── Side-by-side: raw xGA vs Suppression Score ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
fig.suptitle('Raw xGA vs Shot Suppression Score by Team\n(Lower = better defence)', fontsize=13, y=1.01)

sorted_ss = team_merged.sort_values('suppression_score')
teams_sorted = sorted_ss['contestant_id'].astype(str)

# Raw xGA
ax1 = axes[0]
ax1.barh(teams_sorted, sorted_ss['xg_conceded'], color=RED, alpha=0.8, edgecolor='white', linewidth=0.4)
ax1.set_xlabel('Total xGA (raw)')
ax1.set_title('Raw xGA')
ax1.grid(True, axis='x')
ax1.invert_xaxis()

# Suppression Score
ax2 = axes[1]
ax2.barh(teams_sorted, sorted_ss['suppression_score'], color=TEAL, alpha=0.8, edgecolor='white', linewidth=0.4)
ax2.set_xlabel(f'Suppression Score (Non-Blocked xGA + {LAMBDA}×xG_Blocked)')
ax2.set_title(f'Suppression Score (λ={LAMBDA})')
ax2.grid(True, axis='x')

plt.tight_layout()
plt.show()

print('\nNote: Teams where ranking DIFFERS between xGA and Suppression Score')
print('are teams whose block rate significantly changes their defensive picture.')

---
## Section 7 — Shot-Level P(block) Analysis

Which shot characteristics most predict whether a shot is blocked? We use SHAP values to interpret the model, then examine P(block) by archetype, distance, and pitch location.

In [ ]:
# ── SHAP summary plot (top 10 features) ───────────────────────────────────────
explainer   = shap.TreeExplainer(block_model)
X_sample    = pd.DataFrame(X_all[:500], columns=BLOCK_FEATURES)  # sample for speed
shap_values = explainer.shap_values(X_sample)

# SHAP in dark style
plt.rcParams['figure.facecolor'] = NAVY
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_sample,
    feature_names=BLOCK_FEATURES,
    max_display=10,
    plot_type='bar',
    color=TEAL,
    show=False
)
plt.title('SHAP Feature Importance — Block Probability Model', color='white')
plt.tight_layout()
plt.show()

In [ ]:
# ── P(block) by archetype ─────────────────────────────────────────────────────
if 'archetype' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    arch_order = df.groupby('archetype')['p_block'].median().sort_values(ascending=False).index
    sns.violinplot(data=df, x='archetype', y='p_block', order=arch_order,
                   palette=[TEAL, GOLD, RED, GREEN, NAVY, GREY],
                   inner='quartile', cut=0, ax=ax)
    ax.set_xlabel('Shot Archetype')
    ax.set_ylabel('P(block)')
    ax.set_title('P(block) Distribution by Shot Archetype')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print('archetype column not present — skipping violin plot.')

In [ ]:
# ── P(block) by distance zone ─────────────────────────────────────────────────
df['distance_bin'] = pd.cut(df['distance'], bins=10)
dist_pb = df.groupby('distance_bin')['p_block'].mean().reset_index()
dist_pb['dist_mid'] = dist_pb['distance_bin'].apply(lambda x: x.mid)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(dist_pb['dist_mid'], dist_pb['p_block'],
           color=GOLD, s=80, edgecolors='white', linewidth=0.6, zorder=3)
ax.plot(dist_pb['dist_mid'], dist_pb['p_block'],
        color=TEAL, linewidth=2, alpha=0.7)
ax.set_xlabel('Distance from Goal (yards)')
ax.set_ylabel('Mean P(block)')
ax.set_title('P(block) by Distance Zone')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ── P(block) heatmap on pitch ─────────────────────────────────────────────────
if 'x' in df.columns and 'y' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor(NAVY)
    ax.set_facecolor('#111d2b')

    # 2D weighted histogram
    h, xedges, yedges = np.histogram2d(
        df['x'], df['y'],
        bins=[30, 20],
        range=[[0, 105], [0, 68]],
        weights=df['p_block']
    )
    counts, _, _ = np.histogram2d(df['x'], df['y'], bins=[30, 20], range=[[0, 105], [0, 68]])
    with np.errstate(invalid='ignore'):
        h_norm = np.where(counts > 0, h / counts, 0)

    im = ax.imshow(
        h_norm.T,
        origin='lower',
        extent=[0, 105, 0, 68],
        aspect='auto',
        cmap='YlOrRd',
        alpha=0.85,
        vmin=0, vmax=h_norm.max()
    )

    # Pitch markings
    ax.plot([0, 0, 105, 105, 0], [0, 68, 68, 0, 0], color='white', linewidth=1.5)
    ax.plot([52.5, 52.5], [0, 68],   color='white', linewidth=1)
    centre = plt.Circle((52.5, 34), 9.15, fill=False, color='white', linewidth=1)
    ax.add_patch(centre)
    # Penalty areas
    ax.add_patch(plt.Rectangle((0,  13.84), 16.5, 40.32, fill=False, edgecolor='white', linewidth=1))
    ax.add_patch(plt.Rectangle((88.5, 13.84), 16.5, 40.32, fill=False, edgecolor='white', linewidth=1))
    # Goals
    ax.add_patch(plt.Rectangle((0, 30.34), -2, 7.32, fill=False, edgecolor='white', linewidth=1.5))
    ax.add_patch(plt.Rectangle((105, 30.34), 2, 7.32, fill=False, edgecolor='white', linewidth=1.5))

    cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    cbar.set_label('Mean P(block)', color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

    ax.set_xlim(-3, 108)
    ax.set_ylim(-2, 70)
    ax.set_xlabel('Pitch Length (x)')
    ax.set_ylabel('Pitch Width (y)')
    ax.set_title('P(block) Heatmap — Mean Block Probability by Pitch Zone')
    plt.tight_layout()
    plt.show()
else:
    print('x/y columns not available — skipping pitch heatmap.')

---
## Section 8 — Dangerous Blocks (High xG Shots That Were Blocked)

These represent the **highest individual defensive value** in the dataset — shots with xG > 0.15 that were blocked. Each of these events is an outfield player preventing a near-certain goal.

In [ ]:
# ── Top 50 most dangerous blocked shots ───────────────────────────────────────
dangerous_blocks = (
    df[(df['is_blocked'] == 1) & (df['xg'] > 0.15)]
    .sort_values('xg', ascending=False)
)

print(f'Shots with xG > 0.15 that were blocked: {len(dangerous_blocks):,}')

display_cols = ['contestant_id', 'xg', 'p_block', 'xgs_contrib',
                'distance', 'is_header', 'is_big_chance', 'archetype'] \
    if 'archetype' in df.columns else \
    ['contestant_id', 'xg', 'p_block', 'xgs_contrib', 'distance', 'is_header', 'is_big_chance']

# Add player_name and match_file if available
for col in ['player_name', 'match_file']:
    if col in df.columns and col not in display_cols:
        display_cols.insert(1, col)

display_cols = [c for c in display_cols if c in df.columns]
display(dangerous_blocks[display_cols].head(50).round(3))

In [ ]:
# ── Per-team count of dangerous blocks ────────────────────────────────────────
dangerous_by_team = (
    dangerous_blocks.groupby('contestant_id')
    .agg(dangerous_blocks_count=('xg', 'count'),
         dangerous_xg_blocked  =('xg', 'sum'))
    .reset_index()
    .sort_values('dangerous_xg_blocked', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 6))
sorted_db = dangerous_by_team.sort_values('dangerous_xg_blocked')
ax.barh(sorted_db['contestant_id'].astype(str),
        sorted_db['dangerous_xg_blocked'],
        color=GOLD, edgecolor='white', linewidth=0.4, alpha=0.9)

for i, (_, row) in enumerate(sorted_db.iterrows()):
    ax.text(row['dangerous_xg_blocked'] + 0.1, i,
            f'{row["dangerous_blocks_count"]} blocks',
            va='center', ha='left', fontsize=8, color=GREY)

ax.set_xlabel('xG Blocked on Shots with xG > 0.15')
ax.set_title('Dangerous Blocks (xG > 0.15) by Team\n(Label = count of dangerous blocks)')
ax.grid(True, axis='x')
plt.tight_layout()
plt.show()

---
## Section 9 — Summary Table & Ranking

Final merged team table with all metrics. **Ranked by Suppression Score (lower = better)**.

In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
summary = team_merged[[
    'contestant_id', 'rank', 'shots_faced', 'goals_conceded',
    'xg_conceded', 'block_rate', 'xg_blocked',
    'bae', 'xgs_plus', 'suppression_score'
]].sort_values('rank').round(3)

summary.columns = [
    'Team', 'Rank', 'Shots Faced', 'Goals Conceded',
    'xGA', 'Block Rate', 'xG Blocked',
    'BAE', 'xGS+', 'Suppression Score'
]

def colour_suppression(val):
    """Colour-code suppression score: green=low (good), red=high (bad)."""
    if pd.isna(val):
        return ''
    norm = (val - summary['Suppression Score'].min()) / \
           (summary['Suppression Score'].max() - summary['Suppression Score'].min() + 1e-9)
    r = int(224 * norm + 31 * (1 - norm))
    g = int(82  * norm + 185 * (1 - norm))
    b = int(82  * norm + 120 * (1 - norm))
    return f'background-color: rgb({r},{g},{b}); color: white'

display(
    summary.style
    .applymap(colour_suppression, subset=['Suppression Score'])
    .background_gradient(subset=['BAE'], cmap='RdYlGn')
    .background_gradient(subset=['xGS+'], cmap='RdYlGn')
    .format({
        'xGA': '{:.2f}', 'Block Rate': '{:.1%}', 'xG Blocked': '{:.2f}',
        'BAE': '{:+.2f}', 'xGS+': '{:+.2f}', 'Suppression Score': '{:.2f}'
    })
)

In [ ]:
# ── Ranking divergence: which teams change rank most? ─────────────────────────
summary['xGA_rank'] = summary['xGA'].rank().astype(int)
summary['rank_change'] = summary['xGA_rank'] - summary['Rank']

fig, ax = plt.subplots(figsize=(10, 6))
sorted_rc = summary.sort_values('rank_change')
colours   = [GREEN if v > 0 else RED if v < 0 else GREY for v in sorted_rc['rank_change']]

bars = ax.barh(sorted_rc['Team'].astype(str), sorted_rc['rank_change'],
               color=colours, edgecolor='white', linewidth=0.4, alpha=0.9)
ax.axvline(0, color='white', linewidth=1.2)

for bar, val in zip(bars, sorted_rc['rank_change']):
    offset = 0.1 if val >= 0 else -0.1
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+d}', va='center', ha='left' if val >= 0 else 'right',
            fontsize=9, color=GREY)

ax.set_xlabel('Rank Change (Suppression Score vs Raw xGA rank)')
ax.set_title('Ranking Divergence: Raw xGA vs Suppression Score\nPositive = better ranked by Suppression Score')
ax.grid(True, axis='x')

up   = mpatches.Patch(color=GREEN, label='Better rank with Suppression Score')
down = mpatches.Patch(color=RED,   label='Worse rank with Suppression Score')
ax.legend(handles=[up, down])

plt.tight_layout()
plt.show()

---
## Section 10 — Save Outputs

In [ ]:
# ── Shot-level output ─────────────────────────────────────────────────────────
shot_output_cols = ['contestant_id', 'xg', 'is_blocked', 'p_block',
                    'expected_suppression', 'actual_suppression', 'xgs_contrib']

# Add optional columns if present
for col in ['match_file', 'player_name']:
    if col in df.columns:
        shot_output_cols.insert(0, col)

df[shot_output_cols].to_csv('shot_suppression_shots.csv', index=False)
print(f'Saved shot_suppression_shots.csv  ({len(df):,} rows)')

# ── Team-level output ─────────────────────────────────────────────────────────
team_merged.to_csv('team_suppression_scores.csv', index=False)
print(f'Saved team_suppression_scores.csv ({len(team_merged)} teams)')

print('\nAll outputs saved successfully.')